<a href="https://colab.research.google.com/github/modelpredection-glitch/injector-website/blob/main/RUL_Estimator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, brentq
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ────────────────────────────────────────────────────────────────
# SECTION 1 — WEIBULL MLE FITTING
# Run once with your real fleet odometer data.
# Returns BETA and ETA used in all RUL calculations below.
# ────────────────────────────────────────────────────────────────

def fit_weibull_mle(failure_km, suspension_km):
    """
    Fit Weibull β and η by Maximum Likelihood Estimation.
    Incorporates both complete failures and right-censored suspensions.

    Parameters
    ----------
    failure_km    : list — odometer (km) at confirmed injector failure
                    (your 5 faulty vehicles)
    suspension_km : list — current odometer of healthy vehicles
                    (your 10 healthy vehicles — right-censored)

    Returns
    -------
    beta : float — Cox-Snell bias-corrected shape parameter
    eta  : float — scale parameter (characteristic life in km)
    """
    failure_km    = np.array(failure_km,    dtype=float)
    suspension_km = np.array(suspension_km, dtype=float)
    n             = len(failure_km)

    def neg_log_likelihood(params):
        b, e = params
        if b <= 0 or e <= 0:
            return 1e10
        ll  = np.sum(np.log(b/e) + (b-1)*np.log(failure_km/e) - (failure_km/e)**b)
        ll += np.sum(-(suspension_km/e)**b)
        return -ll

    res = minimize(neg_log_likelihood,
                   x0=[2.0, np.mean(failure_km)],
                   method='L-BFGS-B',
                   bounds=[(0.01, 50), (1, 1e7)])

    if not res.success:
        raise RuntimeError(f"MLE did not converge: {res.message}")

    beta_mle, eta = res.x

    # Cox-Snell bias correction for n < 20 complete failures
    beta = beta_mle * (1 - (0.241 + 0.328/n)) if n < 20 else beta_mle

    print(f"\n{'='*55}")
    print(f"  WEIBULL MLE RESULTS")
    print(f"{'='*55}")
    print(f"  Beta (raw MLE)       : {beta_mle:.4f}")
    print(f"  Beta (bias-corrected): {beta:.4f}")
    print(f"  Eta  (char. life)    : {eta:,.0f} km")
    print(f"  Complete failures    : {n}")
    print(f"  Right-censored       : {len(suspension_km)}")
    t_peak = eta * ((beta-1)/beta)**(1/beta)
    print(f"  PDF mode (t_peak)    : {t_peak:,.0f} km")
    print(f"  Failure regime       : {'Severe wear-out' if beta > 4 else 'Wear-out'}")
    print(f"{'='*55}\n")
    return beta, eta


In [ ]:
# ────────────────────────────────────────────────────────────────
# SECTION 2 — WEIBULL PDF HELPERS
# ────────────────────────────────────────────────────────────────

def weibull_pdf(t, beta, eta):
    """Weibull Probability Density Function f(t)."""
    t = np.asarray(t, dtype=float)
    return (beta/eta) * (t/eta)**(beta-1) * np.exp(-(t/eta)**beta)


def pdf_mode(beta, eta):
    """
    t_peak = η × ((β-1)/β)^(1/β)
    The odometer at which the PDF is maximum — the most likely
    failure point in the fleet population. Requires β > 1.
    """
    if beta <= 1:
        raise ValueError("PDF mode requires β > 1.")
    return eta * ((beta - 1) / beta)**(1 / beta)


def normalised_pdf(t, beta, eta):
    """
    f_norm(t) = f(t) / f(t_peak)
    Scales the PDF so its maximum value is 1.
    This allows us to directly compare it with the ML probability p.
    """
    t_pk = pdf_mode(beta, eta)
    f_pk = weibull_pdf(t_pk, beta, eta)
    return weibull_pdf(t, beta, eta) / f_pk


def solve_t_current(p, beta, eta):
    """
    Solve  f_norm(t_current) = p  on the LEFT side of the PDF
    (t_current < t_peak).

    Interpretation:
      The ML model outputs a failure probability p ∈ [0,1].
      We treat this as a normalised failure density level.
      On the left side of the PDF, the normalised density rises
      from 0 (at t=0) to 1 (at t=t_peak).
      Solving for t_current gives us WHERE on the Weibull failure
      curve the vehicle is currently positioned based on its
      ML-predicted health state.

      Higher p → larger t_current → closer to t_peak → shorter RUL ✓
    """
    p      = np.clip(p, 1e-6, 1 - 1e-6)
    t_pk   = pdf_mode(beta, eta)
    f_pk   = weibull_pdf(t_pk, beta, eta)
    t_lo   = eta * 1e-4   # near zero — f_norm ≈ 0 here

    def objective(t):
        return weibull_pdf(t, beta, eta) / f_pk - p

    # If f_norm at t_peak-epsilon is < p, the left side never reaches p
    # This means the vehicle is already past the peak → overdue
    if objective(t_pk - 1e-6) < 0:
        return t_pk   # signal: at or past peak

    try:
        return brentq(objective, t_lo, t_pk - 1e-6, xtol=0.1)
    except ValueError:
        return t_pk


In [ ]:
# ────────────────────────────────────────────────────────────────
# SECTION 3 — CORE RUL FUNCTION (PDF-based)
# ────────────────────────────────────────────────────────────────

def compute_rul(failure_prob, confidence, current_km,
                beta, eta,
                safety_margin        = 0.15,
                fault_prob_threshold = 0.65,
                confidence_threshold = 0.75):
    """
    Compute Remaining Useful Life for one vehicle using the
    Weibull PDF method.

    PDF-BASED RUL LOGIC
    ───────────────────
    t_peak    = η × ((β-1)/β)^(1/β)
                Fixed fleet mode — the most likely failure odometer.
                Same for every vehicle.

    f_norm(t) = f(t) / f_peak
                Normalised PDF, scaled to [0, 1].

    t_current = solution of  f_norm(t) = p  on the LEFT side
                WHERE the vehicle sits on the PDF curve right now.
                Higher p → larger t_current → closer to t_peak.

    RUL_raw   = t_peak − t_current
                Remaining distance to the peak failure zone.
                Higher p → smaller gap → shorter RUL ✓

    RUL       = RUL_raw × (1 − safety_margin)

    Overdue   : when p is so high that t_current ≥ t_peak

    Parameters
    ----------
    failure_prob         : float [0-1]  ML failure probability from XGBoost
    confidence           : float [0-1]  calibrated model confidence
    current_km           : float        current odometer reading (km)
    beta                 : float        Weibull shape (from fit_weibull_mle)
    eta                  : float        Weibull scale (from fit_weibull_mle)
    safety_margin        : float        default 0.15 (15%)
    fault_prob_threshold : float        CBM alpha = 0.65
    confidence_threshold : float        CBM gamma = 0.75

    Returns dict with keys:
        t_peak, t_current, rul_raw, rul_adjusted,
        maintenance_due, urgency, action, already_overdue
    """

    t_pk   = pdf_mode(beta, eta)
    t_curr = solve_t_current(failure_prob, beta, eta)

    rul_raw  = t_pk - t_curr
    overdue  = rul_raw <= 0

    if overdue:
        rul_adj = 0.0
        maint   = float(current_km)
    else:
        rul_adj = rul_raw * (1 - safety_margin)
        maint   = current_km + rul_adj

    # Urgency classification
    if failure_prob >= 0.70 or overdue:
        urgency = "CRITICAL"
    elif failure_prob >= 0.40:
        urgency = "HIGH"
    elif failure_prob >= 0.20:
        urgency = "MEDIUM"
    else:
        urgency = "LOW"

    # CBM action
    p_flag = failure_prob >= fault_prob_threshold
    c_flag = confidence  >= confidence_threshold

    if overdue:
        action = "Vehicle is at or past peak failure zone. IMMEDIATE maintenance required."
    elif p_flag and c_flag:
        action = "Fault detected with HIGH confidence. Schedule maintenance immediately."
    elif p_flag and not c_flag:
        action = "Fault detected with LOW confidence. Inspect within 2 operating cycles."
    elif not p_flag and not c_flag:
        action = "No fault detected. Continue normal CBM monitoring."
    else:
        action = (f"Healthy, HIGH confidence. "
                  f"Next service at {maint:,.0f} km "
                  f"({rul_adj:,.0f} km remaining).")

    return {
        "t_peak"         : round(t_pk,             0),
        "t_current"      : round(t_curr,            0),
        "rul_raw"        : round(max(rul_raw, 0),   0),
        "rul_adjusted"   : round(rul_adj,            0),
        "maintenance_due": round(maint,              0),
        "urgency"        : urgency,
        "action"         : action,
        "already_overdue": overdue
    }


In [ ]:
# ────────────────────────────────────────────────────────────────
# SECTION 4 — FLEET SCHEDULE
# ────────────────────────────────────────────────────────────────

def compute_fleet_rul(vehicles_df, beta, eta, safety_margin=0.15):
    """
    Run compute_rul() for every vehicle and return a sorted
    maintenance schedule DataFrame.

    vehicles_df must have columns:
        vehicle_id, current_km, failure_prob (0-1), confidence (0-1)
    """
    rows = []
    for _, v in vehicles_df.iterrows():
        r = compute_rul(
            failure_prob   = v["failure_prob"],
            confidence     = v["confidence"],
            current_km     = v["current_km"],
            beta           = beta,
            eta            = eta,
            safety_margin  = safety_margin
        )
        rows.append({
            "vehicle_id"   : v["vehicle_id"],
            "current_km"   : v["current_km"],
            "failure_%"    : round(v["failure_prob"]*100, 1),
            "confidence_%" : round(v["confidence"]*100,   1),
            "t_peak_km"    : r["t_peak"],
            "t_current_km" : r["t_current"],
            "rul_raw_km"   : r["rul_raw"],
            "rul_adj_km"   : r["rul_adjusted"],
            "maint_due_km" : r["maintenance_due"],
            "urgency"      : r["urgency"],
            "overdue"      : r["already_overdue"],
            "action"       : r["action"]
        })

    df = pd.DataFrame(rows)
    rank = {"CRITICAL":0,"HIGH":1,"MEDIUM":2,"LOW":3}
    df["_r"] = df["urgency"].map(rank)
    df = df.sort_values(["overdue","_r","rul_adj_km"],
                        ascending=[False,True,True]
                        ).drop(columns="_r").reset_index(drop=True)
    return df


In [ ]:
# ────────────────────────────────────────────────────────────────
# SECTION 5 — check_with_rul() FOR COLAB (drop-in replacement)
# ────────────────────────────────────────────────────────────────

def check_with_rul(df, base_model, calibrated_model, colu,
                   vehicle_id, current_km, beta, eta,
                   safety_margin=0.15):
    """Updated check() function — ML prediction + PDF-based RUL."""
    X            = df[colu]
    failure_prob = base_model.predict_proba(X)[:, 1].mean()
    confidence   = calibrated_model.predict_proba(X).max(axis=1).mean()

    r = compute_rul(failure_prob, confidence, current_km,
                    beta, eta, safety_margin)

    print(f"\n{'='*62}")
    print(f"  VEHICLE : {vehicle_id}")
    print(f"{'='*62}")
    print(f"  Current odometer          : {current_km:>10,.0f} km")
    print(f"  Failure probability (p)   : {failure_prob*100:>9.2f} %")
    print(f"  Model confidence          : {confidence*100:>9.2f} %")
    print(f"  Weibull PDF mode (t_peak) : {r['t_peak']:>10,.0f} km  [fixed]")
    print(f"  Vehicle position (t_curr) : {r['t_current']:>10,.0f} km")
    print(f"  RUL raw (t_peak-t_curr)   : {r['rul_raw']:>10,.0f} km")
    print(f"  RUL ({int(safety_margin*100)}% margin applied)  : {r['rul_adjusted']:>10,.0f} km")
    print(f"  Maintenance due at        : {r['maintenance_due']:>10,.0f} km")
    print(f"  Urgency                   : {r['urgency']}")
    print(f"  Action                    : {r['action']}")
    print(f"{'='*62}")

    return failure_prob, confidence, r


In [ ]:
# ────────────────────────────────────────────────────────────────
# SECTION 6 — calculate_rul_streamlit() FOR app.py
# ────────────────────────────────────────────────────────────────

def calculate_rul_streamlit(failure_percent, confidence_percent, current_km,
                             beta=7.8616, eta=57308.0,
                             safety_margin=0.15,
                             fault_prob_threshold=0.65,
                             confidence_threshold=0.75):
    """
    Streamlit-compatible wrapper for calculate_rul().
    Inputs in percent (0-100); returns same 4-tuple as before.
    """
    r = compute_rul(
        failure_prob         = failure_percent   / 100.0,
        confidence           = confidence_percent / 100.0,
        current_km           = current_km,
        beta                 = beta,
        eta                  = eta,
        safety_margin        = safety_margin,
        fault_prob_threshold = fault_prob_threshold,
        confidence_threshold = confidence_threshold
    )
    return r["rul_adjusted"], r["action"], r["maintenance_due"], r["urgency"]


# ────────────────────────────────────────────────────────────────
# SECTION 7 — VERIFICATION
# Run this file directly to confirm higher p → shorter RUL
# ────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # ── Fit Weibull from fleet data ──────────────────────────────
    failure_km    = [
        78200,
      62400,
      71800,
      64500,
      77300]
    suspension_km = [32000, 41000, 38500, 29000, 55000,
                     47000, 36000, 43500, 51000, 28000]

    BETA, ETA = fit_weibull_mle(failure_km, suspension_km)

    t_pk = pdf_mode(BETA, ETA)
    print(f"PDF mode (t_peak)  = {t_pk:,.0f} km  ← peak failure density")
    print(f"  (failures cluster most densely at this odometer)\n")

    # ── Fleet example ────────────────────────────────────────────
    fleet = pd.DataFrame({
        "vehicle_id"  : ["V-H01","V-H02","V-H03","V-H04","V-H05"],
        "current_km"  : [32000, 41000, 38500, 55000, 47000],
        "failure_prob": [0.72,  0.55,  0.35,  0.88,  0.25],
        "confidence"  : [0.81,  0.72,  0.68,  0.94,  0.60],
    })

    schedule = compute_fleet_rul(fleet, BETA, ETA)
    print("FLEET MAINTENANCE SCHEDULE")
    print(schedule[["vehicle_id","failure_%","t_current_km",
                    "t_peak_km","rul_adj_km","maint_due_km","urgency"
                    ]].to_string(index=False))

    print("\nVERIFICATION — sorted by failure probability:")
    v = schedule.sort_values("failure_%", ascending=False)[
            ["vehicle_id","failure_%","rul_adj_km"]]
    print(v.to_string(index=False))
    ruls = list(v["rul_adj_km"])
    mono = all(ruls[i] >= ruls[i+1] for i in range(len(ruls)-1))
    print(f"\n↑ Higher probability → shorter RUL: {mono} ✓")


  WEIBULL MLE RESULTS
  Beta (raw MLE)       : 13.5122
  Beta (bias-corrected): 9.3693
  Eta  (char. life)    : 73,843 km
  Complete failures    : 5
  Right-censored       : 10
  PDF mode (t_peak)    : 72,959 km
  Failure regime       : Severe wear-out

PDF mode (t_peak)  = 72,959 km  ← peak failure density
  (failures cluster most densely at this odometer)

FLEET MAINTENANCE SCHEDULE
vehicle_id  failure_%  t_current_km  t_peak_km  rul_adj_km  maint_due_km  urgency
     V-H04       88.0       68526.0    72959.0      3768.0       58768.0 CRITICAL
     V-H01       72.0       65574.0    72959.0      6277.0       38277.0 CRITICAL
     V-H02       55.0       62632.0    72959.0      8778.0       49778.0     HIGH
     V-H03       35.0       58646.0    72959.0     12166.0       50666.0   MEDIUM
     V-H05       25.0       56069.0    72959.0     14357.0       61357.0   MEDIUM

VERIFICATION — sorted by failure probability:
vehicle_id  failure_%  rul_adj_km
     V-H04       88.0      3768.0
    

In [ ]:
!pip install scipy --quiet
import numpy as np
import pandas as pd
import pickle
import joblib
import warnings
from scipy.optimize import minimize, brentq

warnings.filterwarnings("ignore")
print("✅ Libraries ready.")


✅ Libraries ready.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

with open("/content/drive/MyDrive/Data/base_model.pkl", "rb") as f:
    base_model = pickle.load(f)

calibrated_model = joblib.load("/content/drive/MyDrive/Data/calibrated_model.pkl")

FEATURES = [
    'Desired Fuel Injection Quantity',
    'Desired Fuel Rail Pressure (FRP)',
    'Desired Mass Air Flow (MAF)',
    'Fuel Rail Pressure (FRP)',
    'Mass Air Flow (MAF)',
    'Main Fuel Injection Quantity',
    'Pre Fuel Injection Quantity',
    'Engine Speed',
    'Boost Pressure'
]

print("✅ Models loaded.")
print(f"   Required CSV columns: {len(FEATURES)}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Models loaded.
   Required CSV columns: 9


In [ ]:

# ──────────────────────────────────────────────────────────────────
# FUNCTION 1: fit_weibull_mle
# Fits β and η from fleet odometer data using MLE.
# Run once in Cell 5. BETA and ETA do not change for new vehicles.
# ──────────────────────────────────────────────────────────────────
def fit_weibull_mle(failure_km, suspension_km):
    failure_km    = np.array(failure_km,    dtype=float)
    suspension_km = np.array(suspension_km, dtype=float)
    n             = len(failure_km)

    def neg_ll(params):
        b, e = params
        if b <= 0 or e <= 0: return 1e10
        ll  = np.sum(np.log(b/e) + (b-1)*np.log(failure_km/e) - (failure_km/e)**b)
        ll += np.sum(-(suspension_km/e)**b)
        return -ll

    res = minimize(neg_ll, x0=[2.0, np.mean(failure_km)],
                   method='L-BFGS-B', bounds=[(0.01,50),(1,1e7)])
    if not res.success:
        raise RuntimeError(f"MLE did not converge: {res.message}")

    b_mle, eta = res.x
    beta = b_mle * (1 - (0.241 + 0.328/n)) if n < 20 else b_mle

    t_peak = eta * ((beta-1)/beta)**(1/beta)
    print(f"\n{'='*55}")
    print(f"  WEIBULL PARAMETERS — COPY INTO app.py IF NEEDED")
    print(f"{'='*55}")
    print(f"  BETA     = {beta:.4f}")
    print(f"  ETA      = {eta:,.2f} km")
    print(f"  t_peak   = {t_peak:,.0f} km  ← PDF mode (peak failure zone)")
    print(f"{'='*55}\n")
    return beta, eta


# ──────────────────────────────────────────────────────────────────
# FUNCTION 2: weibull_pdf
# ──────────────────────────────────────────────────────────────────
def weibull_pdf(t, beta, eta):
    return (beta/eta) * (t/eta)**(beta-1) * np.exp(-(t/eta)**beta)


# ──────────────────────────────────────────────────────────────────
# FUNCTION 3: solve_t_current
#
# Finds WHERE on the Weibull PDF curve the vehicle sits right now.
#
# HOW IT WORKS:
#   The Weibull PDF f(t) rises from 0, peaks at t_peak,
#   then falls back to 0. We normalise it: f_norm(t) = f(t)/f_peak,
#   so it ranges from 0 to 1.
#
#   The ML model outputs probability p ∈ [0,1].
#   We find t_current by solving f_norm(t_current) = p on the LEFT
#   side of the peak (the degradation approach zone).
#
#   As the vehicle degrades: p rises → t_current moves toward t_peak
#   → the gap (t_peak - t_current) shrinks → RUL gets shorter. ✓
# ──────────────────────────────────────────────────────────────────
def solve_t_current(p, beta, eta):
    p      = np.clip(p, 1e-6, 1-1e-6)
    t_peak = eta * ((beta-1)/beta)**(1/beta)
    f_peak = weibull_pdf(t_peak, beta, eta)

    def objective(t):
        return weibull_pdf(t, beta, eta) / f_peak - p

    # If the left side never reaches p, vehicle is at/past peak → overdue
    if objective(t_peak - 1e-6) < 0:
        return t_peak

    try:
        return brentq(objective, eta*1e-4, t_peak - 1e-6, xtol=0.1)
    except ValueError:
        return t_peak


# ──────────────────────────────────────────────────────────────────
# FUNCTION 4: compute_rul
#
# Core RUL calculation — PDF-based.
#
# FORMULA SUMMARY:
#   t_peak    = η × ((β-1)/β)^(1/β)          ← peak failure density
#   t_current = solve f_norm(t) = p (left)    ← vehicle's current position
#   RUL_raw   = t_peak - t_current            ← gap to peak failure zone
#   RUL       = RUL_raw × (1 - 0.15)          ← with 15% safety margin
#   Maint Due = current_km + RUL              ← odometer to service at
# ──────────────────────────────────────────────────────────────────
def compute_rul(failure_prob, confidence, current_km,
                beta, eta, safety_margin=0.15,
                fault_threshold=0.65, conf_threshold=0.75):

    t_peak = eta * ((beta-1)/beta)**(1/beta)
    t_curr = solve_t_current(failure_prob, beta, eta)

    rul_raw = t_peak - t_curr
    overdue = rul_raw <= 0

    rul_adj = 0.0   if overdue else rul_raw * (1 - safety_margin)
    maint   = float(current_km) if overdue else current_km + rul_adj

    if failure_prob >= 0.70 or overdue:
        urgency = "🔴 CRITICAL"
    elif failure_prob >= 0.40:
        urgency = "🟠 HIGH"
    elif failure_prob >= 0.20:
        urgency = "🟡 MEDIUM"
    else:
        urgency = "🟢 LOW"

    p_flag = failure_prob >= fault_threshold
    c_flag = confidence  >= conf_threshold

    if overdue:
        action = "⛔ Vehicle is at/past peak failure zone. IMMEDIATE maintenance required."
    elif p_flag and c_flag:
        action = "🔴 Fault confirmed, HIGH confidence. Schedule maintenance now."
    elif p_flag and not c_flag:
        action = "🟠 Fault signal, LOW confidence. Inspect within 2 cycles."
    elif not p_flag and not c_flag:
        action = "🟡 No fault. Continue CBM monitoring. Use RUL to plan next service."
    else:
        action = f"🟢 Healthy, HIGH confidence. Next service at {maint:,.0f} km."

    return {
        "t_peak"       : round(t_peak,         0),
        "t_current"    : round(t_curr,          0),
        "rul_raw"      : round(max(rul_raw,0),  0),
        "rul_adjusted" : round(rul_adj,          0),
        "maint_due"    : round(maint,            0),
        "urgency"      : urgency,
        "action"       : action,
        "overdue"      : overdue
    }


# ──────────────────────────────────────────────────────────────────
# FUNCTION 5: predict_rul_from_csv
# Takes a CSV file from any new vehicle and prints the full report.
# ──────────────────────────────────────────────────────────────────
def predict_rul_from_csv(csv_path, vehicle_id, current_km, beta, eta,
                          safety_margin=0.15):

    df = pd.read_csv(csv_path)
    missing = set(FEATURES) - set(df.columns)
    if missing:
        print(f"❌ Missing columns in CSV: {missing}")
        return None

    X            = df[FEATURES]
    failure_prob = base_model.predict_proba(X)[:, 1].mean()
    confidence   = calibrated_model.predict_proba(X).max(axis=1).mean()

    r = compute_rul(failure_prob, confidence, current_km,
                    beta, eta, safety_margin)

    print(f"\n{'═'*65}")
    print(f"  VEHICLE HEALTH REPORT  —  {vehicle_id}")
    print(f"{'═'*65}")
    print(f"\n  ML PREDICTION")
    print(f"  {'ECU rows in CSV':<32}: {len(X):,}")
    print(f"  {'Failure probability (p)':<32}: {failure_prob*100:.2f}%")
    print(f"  {'Model confidence':<32}: {confidence*100:.2f}%")
    print(f"\n  WEIBULL PDF RUL  (β={beta:.4f}, η={eta:,.0f} km)")
    print(f"  {'PDF mode — t_peak [fixed]':<32}: {r['t_peak']:,.0f} km")
    print(f"  {'Vehicle position — t_current':<32}: {r['t_current']:,.0f} km")
    print(f"  {'Gap to peak — RUL_raw':<32}: {r['rul_raw']:,.0f} km")
    print(f"  {'RUL (15% safety margin)':<32}: {r['rul_adjusted']:,.0f} km")
    print(f"  {'Current odometer':<32}: {current_km:,.0f} km")
    print(f"  {'Maintenance due at':<32}: {r['maint_due']:,.0f} km")
    print(f"\n  DECISION")
    print(f"  {'Urgency':<32}: {r['urgency']}")
    print(f"  {'Action':<32}: {r['action']}")
    print(f"{'═'*65}\n")

    return {"vehicle_id":vehicle_id, "current_km":current_km,
            "failure_prob":failure_prob, "confidence":confidence, **r}


# ──────────────────────────────────────────────────────────────────
# FUNCTION 6: predict_multiple_csvs
# Run predict_rul_from_csv on a list of vehicles → sorted schedule.
# ──────────────────────────────────────────────────────────────────
def predict_multiple_csvs(vehicle_list, beta, eta, safety_margin=0.15):
    results = []
    for v in vehicle_list:
        r = predict_rul_from_csv(
            csv_path      = v["csv_path"],
            vehicle_id    = v["vehicle_id"],
            current_km    = v["current_km"],
            beta          = beta,
            eta           = eta,
            safety_margin = safety_margin
        )
        if r: results.append(r)

    if not results: return None

    df = pd.DataFrame(results)
    rank = {"🔴 CRITICAL":0,"🟠 HIGH":1,"🟡 MEDIUM":2,"🟢 LOW":3}
    df["_r"] = df["urgency"].map(rank)
    df = df.sort_values(["overdue","_r","rul_adjusted"],
                        ascending=[False,True,True]
                        ).drop(columns="_r").reset_index(drop=True)
    return df


print("✅ All functions defined. Ready to predict.")



✅ All functions defined. Ready to predict.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 5 — [ONE-TIME SETUP] Fit Weibull Parameters
#
# Replace the km values with YOUR real odometer readings.
# Run this once. BETA and ETA are then used for every new prediction.
# ════════════════════════════════════════════════════════════════════

failure_km = [
    78200,   # Faulty vehicle 1 — km when injector confirmed failed
    62400,   # Faulty vehicle 2
    71800,   # Faulty vehicle 3
    64500,   # Faulty vehicle 4
    77300,   # Faulty vehicle 5
]

suspension_km = [
    32000,   # Healthy vehicle 1  — current odometer TODAY
    41000,   # Healthy vehicle 2
    38500,   # Healthy vehicle 3
    29000,   # Healthy vehicle 4
    55000,   # Healthy vehicle 5
    47000,   # Healthy vehicle 6
    36000,   # Healthy vehicle 7
    43500,   # Healthy vehicle 8
    51000,   # Healthy vehicle 9
    28000,   # Healthy vehicle 10
]

BETA, ETA = fit_weibull_mle(failure_km, suspension_km)

t_peak_global = ETA * ((BETA-1)/BETA)**(1/BETA)
print(f"📌 BETA    = {BETA:.4f}")
print(f"📌 ETA     = {ETA:,.0f} km")
print(f"📌 t_peak  = {t_peak_global:,.0f} km  ← peak failure zone (fixed for all vehicles)")



  WEIBULL PARAMETERS — COPY INTO app.py IF NEEDED
  BETA     = 9.3693
  ETA      = 73,843.13 km
  t_peak   = 72,959 km  ← PDF mode (peak failure zone)

📌 BETA    = 9.3693
📌 ETA     = 73,843 km
📌 t_peak  = 72,959 km  ← peak failure zone (fixed for all vehicles)


In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 6 — Predict RUL for ONE New Vehicle
#
# ▶ Change csv_path, vehicle_id, and current_km for your vehicle.
# ▶ The vehicle does NOT need to be in your training data.
# ════════════════════════════════════════════════════════════════════

# ── Set these 3 values for your vehicle ──────────────────────────
csv_path   = "/content/drive/MyDrive/Data/Test Data/after repair 2.csv"
vehicle_id = "New Vehicle — BA-1-KHA-1234"
current_km = 35000                           # odometer reading

result = predict_rul_from_csv(
    csv_path   = csv_path,
    vehicle_id = vehicle_id,
    current_km = current_km,
    beta       = BETA,
    eta        = ETA
)



═════════════════════════════════════════════════════════════════
  VEHICLE HEALTH REPORT  —  New Vehicle — BA-1-KHA-1234
═════════════════════════════════════════════════════════════════

  ML PREDICTION
  ECU rows in CSV                 : 4,631
  Failure probability (p)         : 17.40%
  Model confidence                : 86.15%

  WEIBULL PDF RUL  (β=9.3693, η=73,843 km)
  PDF mode — t_peak [fixed]       : 72,959 km
  Vehicle position — t_current    : 53,523 km
  Gap to peak — RUL_raw           : 19,436 km
  RUL (15% safety margin)         : 16,520 km
  Current odometer                : 35,000 km
  Maintenance due at              : 51,520 km

  DECISION
  Urgency                         : 🟢 LOW
  Action                          : 🟢 Healthy, HIGH confidence. Next service at 51,520 km.
═════════════════════════════════════════════════════════════════



In [ ]:

# ════════════════════════════════════════════════════════════════════
# CELL 7 — Predict RUL for Multiple Vehicles at Once
#
# ▶ Add each vehicle as a dict with csv_path, vehicle_id, current_km.
# ▶ Results are automatically sorted most urgent first.
# ════════════════════════════════════════════════════════════════════

new_vehicles = [
    {"csv_path": "/content/drive/MyDrive/Data/Test Data/after repair.csv",
     "vehicle_id": "Vehicle A — BA-1-KHA-1234", "current_km": 35000},
    {"csv_path": "/content/drive/MyDrive/Data/Test Data/after repair.csv",
     "vehicle_id": "Vehicle B — BA-2-KHA-5678", "current_km": 52000},
    {"csv_path": "/content/drive/MyDrive/Data/Test Data/bigreko 1 idle.csv",
     "vehicle_id": "Vehicle C — BA-3-KHA-9012", "current_km": 78000},
    # Add more vehicles here...
]

schedule = predict_multiple_csvs(new_vehicles, BETA, ETA)

if schedule is not None:
    print("\n" + "="*75)
    print("  FLEET MAINTENANCE SCHEDULE — most urgent first")
    print("="*75)
    pd.set_option("display.width", 160)
    pd.set_option("display.max_colwidth", 35)
    print(schedule[[
        "vehicle_id", "current_km", "failure_prob",
        "rul_adjusted", "maint_due", "urgency"
    ]].to_string(index=False))

    save_path = "/content/drive/MyDrive/Data/rul_schedule.csv"
    schedule.to_csv(save_path, index=False)
    print(f"\n✅ Saved to: {save_path}")


═════════════════════════════════════════════════════════════════
  VEHICLE HEALTH REPORT  —  Vehicle A — BA-1-KHA-1234
═════════════════════════════════════════════════════════════════

  ML PREDICTION
  ECU rows in CSV                 : 6,298
  Failure probability (p)         : 14.64%
  Model confidence                : 87.55%

  WEIBULL PDF RUL  (β=9.3693, η=73,843 km)
  PDF mode — t_peak [fixed]       : 72,959 km
  Vehicle position — t_current    : 52,371 km
  Gap to peak — RUL_raw           : 20,588 km
  RUL (15% safety margin)         : 17,499 km
  Current odometer                : 35,000 km
  Maintenance due at              : 52,499 km

  DECISION
  Urgency                         : 🟢 LOW
  Action                          : 🟢 Healthy, HIGH confidence. Next service at 52,499 km.
═════════════════════════════════════════════════════════════════


═════════════════════════════════════════════════════════════════
  VEHICLE HEALTH REPORT  —  Vehicle B — BA-2-KHA-5678
═══════════════